# OTTO · Round 05 — Action-pair feature comparison

**134 control → 161 typed / 143 collapsed.** Reuse saved candidate pools and six controls. At most twelve new challenger models. Execute one cell at a time; stop on a failure or pause.

In [1]:
from pathlib import Path
import importlib.util, json, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'transition_features.py').is_file() and (p/'launch.py').is_file()),
            Path.home()/'otto_feature_round05')
if not (ROOT/'launch.py').is_file():
    raise FileNotFoundError('Open the notebook from the extracted otto_feature_round05 folder.')
sys.path.insert(0,str(ROOT))
spec = importlib.util.spec_from_file_location('launch', ROOT/'launch.py')
launch = importlib.util.module_from_spec(spec)
sys.modules['launch'] = launch
spec.loader.exec_module(launch)
STATE={'halted':False, 'completed':[]}
def stage(name):
    if STATE['halted']:
        raise RuntimeError('An earlier stage stopped. Return the ZIP before running another stage.')
    try:
        value=launch.run_stage(name)
        STATE['completed'].append(name)
        return value
    except BaseException:
        STATE['halted']=True
        raise
print('KERNEL_READY', flush=True)
print('NOTEBOOK_PYTHON:',sys.executable)
print('ML_PYTHON_UNCHANGED:',Path.home()/'otto-recommender-system/.venv/bin/python')
print('PACKAGE:',ROOT)

KERNEL_READY
NOTEBOOK_PYTHON: /opt/conda/bin/python
ML_PYTHON_UNCHANGED: /home/sagemaker-user/otto-recommender-system/.venv/bin/python
PACKAGE: /home/sagemaker-user/otto_feature_round05


## Index gate

In [2]:
summary=json.loads((ROOT/'outputs/index_summary.json').read_text())
assert summary['status']=='ROUND05_INDEX_READY'
assert len(summary['partitions'])==16 and summary['study_sessions_excluded']==5120
print('INDEX_GATE_PASSED')

INDEX_GATE_PASSED


## Features — 320-second outer cap
Read certified control features; build only 27 typed plus nine ablation columns, in complete 64-session chunks. First 16 sessions are replayed numerically.

In [3]:
stage('features')

RUNNING features; process cap 320s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round05/outputs/features.log
{"completed": 256, "event": "feature_checkpoints", "stage": "action_pair_features", "total": 4096, "utc": "2026-09-12T03:59:56.312515+00:00"}
{"completed": 512, "event": "feature_checkpoints", "stage": "action_pair_features", "total": 4096, "utc": "2026-09-12T03:59:56.737175+00:00"}
{"completed": 768, "event": "feature_checkpoints", "stage": "action_pair_features", "total": 4096, "utc": "2026-09-12T03:59:57.157580+00:00"}
{"completed": 1024, "event": "feature_checkpoints", "stage": "action_pair_features", "total": 4096, "utc": "2026-09-12T03:59:57.576728+00:00"}
{"completed": 1280, "event": "feature_checkpoints", "stage": "action_pair_features", "total": 4096, "utc": "2026-09-12T03:59:57.993478+00:00"}
{"completed": 1536, "event": "feature_checkpoints", "stage": "action_pair_features", "total": 4096, "utc": "2026-09-12T03:59:58.409526+00:00"}
{"completed": 17

{'phase': 'features', 'exit_code': 0}

## Matched screen — 260-second outer cap
Verify all six old control predictions before any challenger fit. Same chronology, embargo, targets, negatives, capacity and 150 rounds. Save and verify every native model.

In [4]:
stage('screen')

RUNNING screen; process cap 260s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round05/outputs/screen.log
{"event": "all_six_controls_verified_before_new_fits", "utc": "2026-09-12T04:00:16.080495+00:00"}
2026-09-12T04:00:17.716565+00:00 LAUNCHER_HEARTBEAT phase=screen seconds=15.0
{"completed": 0, "elapsed_seconds": 15.0, "event": "heartbeat", "stage": "matched_action_pair_screen", "total": 12, "utc": "2026-09-12T04:00:17.787901+00:00"}
{"arm": "action_pairs", "completed": 1, "event": "model_checkpoint", "fold": 0, "new_fit": 1, "objective": "clicks", "stage": "matched_action_pair_screen", "total": 12, "utc": "2026-09-12T04:00:19.880399+00:00"}
{"arm": "action_pairs", "completed": 2, "event": "model_checkpoint", "fold": 0, "new_fit": 1, "objective": "carts", "stage": "matched_action_pair_screen", "total": 12, "utc": "2026-09-12T04:00:23.718086+00:00"}
{"arm": "action_pairs", "completed": 3, "event": "model_checkpoint", "fold": 0, "new_fit": 1, "objective": "orders", 

{'phase': 'screen', 'exit_code': 0}

## Report — 60-second cap
Recompute from saved per-session statistics. No additional fitting.

In [5]:
stage('report')

RUNNING report; process cap 60s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round05/outputs/report.log
RESULT: ROUND05_REPORT_READY
FINISHED report: 1.2 seconds
RETURN_FILE: /home/sagemaker-user/otto_feature_round05/otto_round05_return.zip


{'phase': 'report', 'exit_code': 0}

## Completion
Save this notebook, then review `03_saved_results.ipynb`. After saving both, use the terminal bundle command to capture their latest bytes.

In [6]:
result=json.loads((ROOT/'outputs/result.json').read_text())
assert result['status']=='ROUND05_SCREEN_COMPLETED'
print('ROUND05_EXECUTION_COMPLETE')
print('Matched primary gain:',result['comparisons']['action_pairs_minus_control_shared']['gain'])
print('This is a fitting-data experiment, not a Kaggle score.')
launch.collect()

ROUND05_EXECUTION_COMPLETE
Matched primary gain: -0.01280409049780129
This is a fitting-data experiment, not a Kaggle score.
RETURN_FILE: /home/sagemaker-user/otto_feature_round05/otto_round05_return.zip


'/home/sagemaker-user/otto_feature_round05/otto_round05_return.zip'